# Household Energy Data - Exploration

In [19]:
%pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


Get data from xlsx file

In [20]:
import os
import pandas as pd

data_path = os.path.join(os.getcwd(), "..", "data", "A.xlsx")
excel_tables = pd.ExcelFile(data_path)
sheet_names = excel_tables.sheet_names

# ['General Information', 'MOBIe EV chargers', 'EVs', 'Load', 'PV', 'BESS', 'Buy Price', 'Sell Price', 'Limits', 'EV1 Buy Price', 'EV2 Buy Price', 'EV1 Load', 'EV2 Load', 'Max EV1 Dis-Charge', 'Max EV2 Dis-Charge', 'EV1 at Home (x)', 'EV2 at Home (x)', 'EV1 at Charging Station (x)', 'EV2 at Charging Station (x)']


Tables:
['General Information', 'MOBIe EV chargers', 'EVs', 'Load', 'PV', 'BESS', 'Buy Price', 'Sell Price', 'Limits', 'EV1 Buy Price', 'EV2 Buy Price', 'EV1 Load', 'EV2 Load', 'Max EV1 Dis-Charge', 'Max EV2 Dis-Charge', 'EV1 at Home (x)', 'EV2 at Home (x)', 'EV1 at Charging Station (x)', 'EV2 at Charging Station (x)']

We now move the data we need into our SQLite DB, table by table.

In [30]:
import sqlite3

conn = sqlite3.connect('energy.db')
cur = conn.cursor()

### Table "General Information"

In [ ]:
# Extract tables into dataframes
general_info_df = pd.read_excel(excel_tables, sheet_name='General Information')

sub_tables = {
    # coordinate mapping example: A5 to C54 = (0, 3) to (3, 254) -> rows (3,254), cols (0,3)
    # calc rows(-2, -1) cols(A=0)
    "Amount": {"rows": (3,254), "cols": (0,3), "columns": ["Player_ID","has_PV","has_ESS"]},
    "Total": {"rows": (2,5), "cols": (4,6), "columns": ["Counter","Count"]},
    "Contractual Power Term": {"rows": (12, 23), "cols": (4,7), "columns": ["CP", "Price/Day", "Units"]},
    "Time of Use Tariff (3)": {"rows": (27, 30), "cols": (4,7), "columns": ["CP", "LTE_20_7", "GT_20_7"]},
    "Time of Use Tariff (4)": {"rows": (33, 36), "cols": (4,8), "columns": ["Super Off-Peak", "Off-Peak", "Mid-Peak", "On-Peak"]},
    "Energy Storage Systems": {"rows": (3, 11), "cols": (10, 17), "columns": ["Type", "Capacity (kW)", "Charge (kW)", "Discharge (kW)", "Efficiency (%)", "Model", "Units"]},
    "Premium Charger EDP": {"rows": (15, 17), "cols": (10, 13), "columns": ["Type", "Capacity (kW)", "Units"]},
    "Regular EV Mpdels": {"rows": (3, 18), "cols": (19, 28), "columns": ["Model ID", "Brand", "Model", "Capacity (kW)", "Charge (kW)", "Discharge (kW)", "Efficiency", "Consumption (Wh/km)", "Units"]},
    "Premium EV Models": {"rows": (22, 32), "cols": (19, 28), "columns": ["Model ID", "Brand", "Model", "Capacity (kW)", "Charge (kW)", "Discharge (kW)", "Efficiency", "Consumption (Wh/km)", "Units"]},
    "EV Departure and Arrivals": {"rows": (4, 32), "cols": (30, 37), "columns": ["Period", "Departure_EV1", "Arrival_EV1", "Departure_EV2", "Arrival_EV2", "Departure_Total", "Arrival_Total"]},
}

for table_name, params in sub_tables.items():
    rows = sub_tables[table_name]["rows"]
    cols = sub_tables[table_name]["cols"]
    sub_tables[table_name]["dataframe"] = general_info_df.iloc[rows[0]:rows[1], cols[0]:cols[1]]
    sub_tables[table_name]["dataframe"].columns = sub_tables[table_name]["columns"]


# print(sub_tables["Amount"]["dataframe"])
# print(sub_tables["Total"]["dataframe"])
# print(sub_tables["Contractual Power Term"]["dataframe"])
# print(sub_tables["Time of Use Tariff (3)"]["dataframe"])
# print(sub_tables["Time of Use Tariff (4)"]["dataframe"])
# print(sub_tables["Energy Storage Systems"]["dataframe"])
# print(sub_tables["Premium Charger EDP"]["dataframe"])
# print(sub_tables["Regular EV Mpdels"]["dataframe"])
# print(sub_tables["Premium EV Models"]["dataframe"])
# print(sub_tables["EV Departure and Arrivals"]["dataframe"])

In [ ]:

# Amount
# -> Players table with data for each PID

general_info_df = pd.read_excel(excel_tables, sheet_name='General Information')
# print(general_info_df)

amount_df = general_info_df.iloc[3:254, 0:3]
amount_df.columns = ["Player_ID", "has_PV", "has_ESS"]

# print(amount_df)

cur.execute("""
CREATE TABLE IF NOT EXISTS Players (
    Player_ID INTEGER PRIMARY KEY,
    has_PV INTEGER,
    has_ESS INTEGER
)
""")

amount_df.to_sql('Players', conn, if_exists='replace', index=False)

conn.commit()

# Total
# -> General Table (we can make as much tables as we want)

total_df = general_info_df.iloc[2:5, 4:6]
total_df.columns = ["Counter", "Count"]

# print(total_df)

cur.execute("""
CREATE TABLE IF NOT EXISTS General_Counters (
    Counter CHAR PRIMARY KEY,
    Count INTEGER
)
""")

total_df.to_sql('General_Counters', conn, if_exists='replace', index=False)

conn.commit()

# Contractual Power Term
# -> assigns price class based on CP (KVA) value

cpt_df = general_info_df.iloc[12:23, 4:7]
cpt_df.columns = ["CP(KVA)", "Price_Per_Day", "Count"]
# print(cpt_df)

cur.execute("""
CREATE TABLE IF NOT EXISTS Contractual_Power_Term (
    CP_KVA INTEGER PRIMARY KEY,
    Price_Per_Day REAL,
    Count INTEGER
)
""")

conn.commit()

# Time of Use Tariff (Triple Price)
# -> assigns tarriff based on time of day and CP (KVA) value

tut_df = general_info_df.iloc[27:30, 4:7]
tut_df.columns = ["CP_KVA", "LESS_OR_EQ_20_7", "MORE_THAN_20_7"]
print(tut_df)

cur.execute("""
CREATE TABLE IF NOT EXISTS Contractual_Power_Term (
    CP_KVA INTEGER PRIMARY KEY,
    LESS_OR_EQ_20_7 REAL,
    MORE_THAN_20_7 REAL
)
""")

conn.commit()

# Time of use Tarrif (Quad Price)
# TBD i dont get it


          CP_KVA LESS_OR_EQ_20_7 MORE_THAN_20_7
27   On-Peak (€)          0.2336         0.2982
28  Mid-Peak (€)           0.171         0.1623
29  Off-Peak (€)          0.1073         0.0953


In [39]:
cur.execute("SELECT * FROM Players LIMIT 10;")
print(cur.fetchall())
print(os.path.abspath('energy.db'))

[(1, 1, 1), (2, 0, 0), (3, 0, 0), (4, 1, 1), (5, 0, 0), (6, 1, 1), (7, 1, 0), (8, 1, 1), (9, 1, 1), (10, 1, 1)]
/Volumes/workspace/Project-Household-Energy-Optimization/src/energy.db


In [28]:
conn.commit()
conn.close()